# Cleaning Dataset 

In [ ]:
import pandas as pd

df = pd.read_csv("Dataset_GPT2_Eng_FIXED.csv")

before = len(df)

df = df.drop_duplicates(subset=["context", "response"])

after = len(df)

print(f"Removed {before - after} duplicate rows")

df.to_csv("Dataset_no_duplicates.csv", index=False)


In [ ]:
df["resp_len"] = df["response"].str.split().apply(len)

df = df[df["resp_len"] >= 4]   # keep responses ≥ 4 words

df = df.drop(columns=["resp_len"])

df.to_csv("Dataset_no_short_answers.csv", index=False)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["context"])

similarity = cosine_similarity(X)

to_remove = set()

threshold = 0.9  # very similar

for i in range(len(similarity)):
    for j in range(i + 1, len(similarity)):
        if similarity[i, j] > threshold:
            to_remove.add(j)

df_clean = df.drop(df.index[list(to_remove)])

df_clean.to_csv("Dataset_no_similar_questions.csv", index=False)


# Dialo GPT

In [ ]:
!pip install -q transformers datasets accelerate torch


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)


In [ ]:
import pandas as pd

# Load with python engine (more tolerant)
df = pd.read_csv(
    "/kaggle/input/dataset-gpt2-eng/Dataset_GPT2_Eng.csv",
    engine="python",
    on_bad_lines="skip"   # skip broken rows
)

# Keep only required columns
df = df[["context", "response"]]

# Drop empty rows
df = df.dropna()

# Save fixed CSV
fixed_path = "/kaggle/working/Dataset_GPT2_Eng_FIXED.csv"
df.to_csv(fixed_path, index=False)

print("Fixed file saved at:", fixed_path)
print("Total rows:", len(df))


In [ ]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/dataset-gpt2-eng/New_dataset.csv"
)

dataset


In [ ]:
def format_conversation(example):
    text = f"Context: {example['context']} <|sep|> Response: {example['response']}"
    return {"text": text}

dataset = dataset.map(format_conversation)


In [ ]:
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# VERY IMPORTANT for DialoGPT
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./dialogpt_finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,  # effective batch size = 16
    learning_rate=5e-5,
    fp16=True,                      # enable if GPU available
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)


In [ ]:
trainer.train()


In [ ]:
trainer.save_model("./dialogpt_finetuned_1")
tokenizer.save_pretrained("./dialogpt_finetuned_1")

print("Fine-tuned model saved!")


# DISTILL GPT

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)


In [ ]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/conversation-dataset/Conversation.csv"
)

dataset


In [ ]:
def format_data(example):
    text = f"Context: {example['context']} <|sep|> Response: {example['answer']}"
    return {"text": text}

dataset = dataset.map(format_data)


In [ ]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 family needs this
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


In [ ]:
def tokenize_fn(example):
    enc = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_dataset = dataset.map(
    tokenize_fn,
    remove_columns=dataset["train"].column_names
)


In [ ]:
tokenized_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/distilgpt2_Conversation",

    # 🔴 DISABLE CHECKPOINT SAVING
    save_strategy="no",

    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    fp16=True,

    logging_steps=100,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer
)


In [ ]:
trainer.train()


In [ ]:
trainer.save_model("/kaggle/working/distilgpt2_Conversation")
tokenizer.save_pretrained("/kaggle/working/distilgpt2_Conversation")


# Saving Model

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/dialogpt_finetuned_1",
    "zip",
    "/kaggle/working/dialogpt_finetuned"
)

print("ZIP created!")


# Testing Models

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/kaggle/working/distilgpt2_finetuned"  # path of trained model

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def chat(question):
    prompt = f"Context: {question} <|sep|> Response:"

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=60,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("Response:")[-1].strip()


In [ ]:
print(chat("who are you ?"))

In [ ]:
chat("Hi")

In [ ]:
chat("Who are you ?")

In [ ]:
chat("Who am i ?")

In [ ]:
chat("what is your name ?")

In [ ]:
chat("what is your name ?")

In [ ]:
chat("what is your name ?")

In [ ]:
chat("What is a seed?")

In [ ]:
chat("what is dog ?")

In [ ]:
chat("What is a root ?")

In [ ]:
chat("what is dog ?")

In [ ]:
chat("what is dog ?")

In [ ]:
chat("what is a mountain ?")

In [ ]:
chat("what is zoo ?")

# HEALTH CHECKUP